In [0]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer

df_clean = spark.read.table("xaids_catalogue.02_silver.cicds2017_cleaned")

fractions = {row["Label"]: 0.8 for row in df_clean.select("Label").distinct().collect()}
train_df = df_clean.sampleBy("Label", fractions=fractions, seed=42)
test_df = df_clean.subtract(train_df)


train_classes = set(r["Label"] for r in train_df.select("Label").distinct().collect())
test_classes = set(r["Label"] for r in test_df.select("Label").distinct().collect())
all_classes = set(r["Label"] for r in df_clean.select("Label").distinct().collect())

missing_in_train = all_classes - train_classes
missing_in_test = all_classes - test_classes
print(f"Classes absentes du train: {missing_in_train}")
print(f"Classes absentes du test: {missing_in_test}")

Classes absentes du train: set()
Classes absentes du test: set()


In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

df_sample_spark = train_df.sample(fraction=0.1, seed=42)

df_pandas = df_sample_spark.toPandas()

numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
df_numeric = df_pandas.select_dtypes(include=numerics)

corr_matrix = df_numeric.corr(method='spearman')

In [0]:
import numpy as np

mask = np.tril(np.ones(corr_matrix.shape), k=0).astype(bool)
corr_upper = corr_matrix.where(~mask)


pairs_df_native = (corr_upper.unstack()
                   .dropna()
                   .reset_index())
pairs_df_native.columns = ['feature_2', 'feature_1', 'correlation']
pairs_df_native = pairs_df_native[pairs_df_native['correlation'].abs() >= 0.9]

pairs_df_native = pairs_df_native.sort_values(by="correlation", key=abs, ascending=False)

display(spark.createDataFrame(pairs_df_native))

feature_2,feature_1,correlation
ECE_Flag_Count,RST_Flag_Count,1.0
Subflow_Fwd_Bytes,Total_Length_of_Fwd_Packets,1.0
Subflow_Bwd_Packets,Total_Backward_Packets,1.0
Subflow_Fwd_Packets,Total_Fwd_Packets,1.0
Avg_Bwd_Segment_Size,Bwd_Packet_Length_Mean,1.0
Avg_Fwd_Segment_Size,Fwd_Packet_Length_Mean,1.0
CWE_Flag_Count,Fwd_URG_Flags,1.0
SYN_Flag_Count,Fwd_PSH_Flags,1.0
Subflow_Bwd_Bytes,Total_Length_of_Bwd_Packets,1.0
Packet_Length_Variance,Packet_Length_Std,0.9999869470755464


In [0]:
from sklearn.feature_selection import mutual_info_classif
import networkx as nx
import pandas as pd

pairs_df_native['abs_corr'] = pairs_df_native['correlation'].abs()

G = nx.Graph()
for _, row in pairs_df_native.iterrows():
    G.add_edge(row['feature_1'], row['feature_2'], weight=row['abs_corr'])

clusters = list(nx.connected_components(G))
all_features = sorted(set(f for c in clusters for f in c))

X = df_pandas[all_features]
y = df_pandas["Label"]

mi_array = mutual_info_classif(X, y, random_state=42)

mi_scores = dict(zip(all_features, mi_array))

cols_to_drop = []

print("\n3. Évaluation des clusters :\n")
for cluster in clusters:
    ranked = sorted(cluster, key=lambda f: mi_scores[f], reverse=True)

    best_feature = ranked[0]
    best_mi_score = mi_scores[best_feature]

    threshold = best_mi_score * 0.90
    
    keep = []
    drop = []

    for f in ranked:
        if mi_scores[f] >= threshold:
            keep.append(f)
        else:
            drop.append(f)
            
    cols_to_drop.extend(drop)

    kept_info = [f"{f} (MI={mi_scores[f]:.4f})" for f in keep]
    dropped_info = [f"{f} (MI={mi_scores[f]:.4f})" for f in drop]
    
    print(f"--- Cluster de {len(cluster)} features ---")
    print(f"Seuil de maintien : >= {threshold:.4f} (Best : {best_mi_score:.4f})")
    print(f"Garde : {kept_info}")
    if drop:
        print(f"Drop  : {dropped_info}\n")
    else:
        print(f"Drop  : Aucun (toutes les features sont performantes)\n")

print(f"===================================================")
print(f"Total des features redondantes à supprimer : {len(cols_to_drop)}")


3. Évaluation des clusters :

--- Cluster de 2 features ---
Seuil de maintien : >= 0.0008 (Best : 0.0009)
Garde : ['ECE_Flag_Count (MI=0.0009)']
Drop  : ['RST_Flag_Count (MI=0.0000)']

--- Cluster de 5 features ---
Seuil de maintien : >= 0.4166 (Best : 0.4629)
Garde : ['Total_Length_of_Fwd_Packets (MI=0.4629)', 'Subflow_Fwd_Bytes (MI=0.4617)', 'Fwd_Packet_Length_Max (MI=0.4288)']
Drop  : ['Fwd_Packet_Length_Mean (MI=0.3764)', 'Avg_Fwd_Segment_Size (MI=0.3764)']

--- Cluster de 6 features ---
Seuil de maintien : >= 0.3305 (Best : 0.3672)
Garde : ['Bwd_Header_Length (MI=0.3672)']
Drop  : ['Bwd_IAT_Max (MI=0.2990)', 'Bwd_IAT_Total (MI=0.2787)', 'Bwd_IAT_Mean (MI=0.2630)', 'Total_Backward_Packets (MI=0.2612)', 'Subflow_Bwd_Packets (MI=0.2606)']

--- Cluster de 6 features ---
Seuil de maintien : >= 0.3730 (Best : 0.4145)
Garde : ['Fwd_IAT_Max (MI=0.4145)', 'Fwd_IAT_Total (MI=0.3933)', 'Fwd_Header_Length (MI=0.3921)', 'Fwd_IAT_Mean (MI=0.3753)']
Drop  : ['Subflow_Fwd_Packets (MI=0.2473)', '

In [0]:
df_fe = df_pandas.copy()

import pandas as pd

regex_pattern = r"(?i)(Monday|Tuesday|Wednesday|Thursday|Friday)"

df_fe["day_of_week_str"] = df_fe["source_file"].str.extract(regex_pattern, expand=False)
frequent_days = df_fe["day_of_week_str"].value_counts().index.tolist()
day_mapping = {day: float(index) for index, day in enumerate(frequent_days)}
df_fe["day_of_week_idx"] = df_fe["day_of_week_str"].map(day_mapping)
idx_fallback = float(len(day_mapping))
df_fe["day_of_week_idx"] = df_fe["day_of_week_idx"].fillna(idx_fallback)


df_fe["fwd_bwd_byte_ratio"] = df_fe["Total_Length_of_Fwd_Packets"] / (df_fe["Total_Length_of_Bwd_Packets"] + 1)
df_fe["fwd_bwd_packet_ratio"] = df_fe["Total_Fwd_Packets"] / (df_fe["Total_Backward_Packets"] + 1)
df_fe["bytes_per_packet_fwd"] = df_fe["Total_Length_of_Fwd_Packets"] / (df_fe["Total_Fwd_Packets"] + 1)
df_fe["bytes_per_packet_bwd"] = df_fe["Total_Length_of_Bwd_Packets"] / (df_fe["Total_Backward_Packets"] + 1)
df_fe["iat_coeff_variation"] = df_fe["Flow_IAT_Std"] / (df_fe["Flow_IAT_Mean"] + 1)

df_fe["flag_density"] = (df_fe["FIN_Flag_Count"] + df_fe["SYN_Flag_Count"] + df_fe["RST_Flag_Count"] +
                         df_fe["PSH_Flag_Count"] + df_fe["ACK_Flag_Count"] + df_fe["URG_Flag_Count"]) / \
                        (df_fe["Total_Fwd_Packets"] + df_fe["Total_Backward_Packets"] + 1)

df_fe["has_no_win_scaling_fwd"] = (df_fe["Init_Win_bytes_forward"] == -1).astype(int)
df_fe["has_no_win_scaling_bwd"] = (df_fe["Init_Win_bytes_backward"] == -1).astype(int)


In [0]:
from sklearn.feature_selection import mutual_info_classif
from pyspark.sql import functions as F
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

candidate_cols = [
    "fwd_bwd_byte_ratio", "fwd_bwd_packet_ratio", "bytes_per_packet_fwd",
    "bytes_per_packet_bwd", "iat_coeff_variation", "flag_density", "day_of_week_idx"
]

print("1. Calcul des scores d'Information Mutuelle (MI)...")

X = df_fe[candidate_cols]
y = df_fe["Label"]

mi_scores = mutual_info_classif(X, y, random_state=42)

result_df = pd.DataFrame({
    "feature": candidate_cols,
    "MI_score": mi_scores
}).sort_values("MI_score", ascending=False)

print("--- Résultats Information Mutuelle ---")
print(result_df.to_string(index=False))
print("\n")

print("2. Vérification des corrélations (Redondance)...")

cols_to_drop_corr = [] 
kept_features = [c for c in df_fe.columns if c not in cols_to_drop_corr + ["Label", "label_idx", "source_file", "day_of_week_str"] and c not in candidate_cols]

for new_col in candidate_cols:
    correlations = {}
    for kept in kept_features:
        if kept != new_col:
            # Calcul direct de la corrélation avec Pandas
            correlations[kept] = df_fe[new_col].corr(df_fe[kept])
            
    # Trouver la feature existante ayant la plus forte corrélation absolue
    max_corr_feature = max(correlations, key=lambda k: abs(correlations[k] or 0))
    max_corr_value = correlations[max_corr_feature] or 0
    
    print(f"{new_col} → corrélation max avec feature existante : {max_corr_feature} ({max_corr_value:.3f})")

1. Calcul des scores d'Information Mutuelle (MI)...
--- Résultats Information Mutuelle ---
             feature  MI_score
  fwd_bwd_byte_ratio  0.534523
bytes_per_packet_bwd  0.490630
bytes_per_packet_fwd  0.457795
        flag_density  0.356129
 iat_coeff_variation  0.333566
fwd_bwd_packet_ratio  0.302265
     day_of_week_idx  0.265257


2. Vérification des corrélations (Redondance)...


/opt/databricks-environments/databricks-ai/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/opt/databricks-environments/databricks-ai/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


fwd_bwd_byte_ratio → corrélation max avec feature existante : Fwd_Packet_Length_Mean (0.235)
fwd_bwd_packet_ratio → corrélation max avec feature existante : Active_Max (0.308)
bytes_per_packet_fwd → corrélation max avec feature existante : Fwd_Packet_Length_Mean (0.991)
bytes_per_packet_bwd → corrélation max avec feature existante : Avg_Bwd_Segment_Size (0.997)
iat_coeff_variation → corrélation max avec feature existante : Total_Fwd_Packets (0.561)
flag_density → corrélation max avec feature existante : ACK_Flag_Count (0.718)
day_of_week_idx → corrélation max avec feature existante : Packet_Length_Std (-0.244)


In [0]:
iat_related_cols = [c for c in train_df.columns if "IAT" in c or "Init_Win" in c]

for c in iat_related_cols:
    neg_one_count = train_df.filter(F.col(c) == -1).count()
    if neg_one_count > 0:
        print(f"{c}: {neg_one_count} valeurs à -1")

Init_Win_bytes_forward: 761240 valeurs à -1
Init_Win_bytes_backward: 1011876 valeurs à -1


In [0]:
import pandas as pd

numeric_cols = df_fe.select_dtypes(include=["number"]).columns.tolist()

excluded_cols = ("Label", "label_idx", "source_file", "ingestion_timestamp")
feature_cols = [c for c in numeric_cols if c not in excluded_cols]

skew_series = df_fe[feature_cols].skew()
min_series = df_fe[feature_cols].min()

group_normal, group_log, group_robust = [], [], []

for c in feature_cols:
    skew = 0 if pd.isna(skew_series[c]) else skew_series[c]
    mn = min_series[c]
    
    if abs(skew) < 1:
        group_normal.append(c)          
    elif mn is not None and mn >= 0:
        group_log.append(c)            
    else:
        group_robust.append(c)          

print(f"Group StandardScaler (quasi-normal): {len(group_normal)} -> {group_normal}")
print(f"Group log1p + StandardScaler (skewed positif): {len(group_log)} -> {group_log}")
print(f"Group RobustScaler (skewed + négatif/outliers): {len(group_robust)} -> {group_robust}")


Group StandardScaler (quasi-normal): 14 -> ['Bwd_PSH_Flags', 'Bwd_URG_Flags', 'PSH_Flag_Count', 'ACK_Flag_Count', 'Fwd_Avg_Bytes/Bulk', 'Fwd_Avg_Packets/Bulk', 'Fwd_Avg_Bulk_Rate', 'Bwd_Avg_Bytes/Bulk', 'Bwd_Avg_Packets/Bulk', 'Bwd_Avg_Bulk_Rate', 'min_seg_size_forward', 'day_of_week_idx', 'has_no_win_scaling_fwd', 'has_no_win_scaling_bwd']
Group log1p + StandardScaler (skewed positif): 70 -> ['Destination_Port', 'Flow_Duration', 'Total_Fwd_Packets', 'Total_Backward_Packets', 'Total_Length_of_Fwd_Packets', 'Total_Length_of_Bwd_Packets', 'Fwd_Packet_Length_Max', 'Fwd_Packet_Length_Min', 'Fwd_Packet_Length_Mean', 'Fwd_Packet_Length_Std', 'Bwd_Packet_Length_Max', 'Bwd_Packet_Length_Min', 'Bwd_Packet_Length_Mean', 'Bwd_Packet_Length_Std', 'Flow_Bytes/s', 'Flow_Packets/s', 'Flow_IAT_Mean', 'Flow_IAT_Std', 'Flow_IAT_Max', 'Flow_IAT_Min', 'Fwd_IAT_Total', 'Fwd_IAT_Mean', 'Fwd_IAT_Std', 'Fwd_IAT_Max', 'Fwd_IAT_Min', 'Bwd_IAT_Total', 'Bwd_IAT_Mean', 'Bwd_IAT_Std', 'Bwd_IAT_Max', 'Bwd_IAT_Min', 